# EcoShield AI — Heavy LightGBM Model

Bu notebook kapsamında IEEE-CIS Fraud Detection veri seti üzerinde
ağır LightGBM modeli geliştirilecektir.

Yapılacak işlemler:

1. Transaction ve identity verilerinin okunması
2. Veri temizleme ve preprocessing
3. Train, validation ve test ayrımı
4. Class imbalance analizi
5. Heavy LightGBM modelinin eğitilmesi
6. Threshold tuning
7. Model metriklerinin hesaplanması
8. Feature importance analizi
9. Model ve sonuçların kaydedilmesi

In [1]:
!pip install -q lightgbm psutil tqdm

In [2]:
# Kullanacağımız kütüphaneler

import os
import gc
import time
import json
import platform
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Python sürümü:", platform.python_version())
print("Pandas sürümü:", pd.__version__)
print("LightGBM sürümü:", lgb.__version__)
print("Toplam RAM:", round(psutil.virtual_memory().total / (1024 ** 3), 2), "GB")
print("Kurulum başarılı.")

Python sürümü: 3.12.13
Pandas sürümü: 2.2.2
LightGBM sürümü: 4.6.0
Toplam RAM: 12.67 GB
Kurulum başarılı.


In [3]:
transaction_path = "/content/drive/MyDrive/EcoShieldAI/train_transaction.csv"
identity_path = "/content/drive/MyDrive/EcoShieldAI/train_identity.csv"

In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/EcoShieldAI/data/raw")

TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IDENTITY_PATH = DATA_DIR / "train_identity.csv"

print("Transaction yolu:", TRANSACTION_PATH)
print("Identity yolu:", IDENTITY_PATH)

print("\nDosya kontrolleri:")
print("train_transaction.csv bulundu mu?", TRANSACTION_PATH.exists())
print("train_identity.csv bulundu mu?", IDENTITY_PATH.exists())

Transaction yolu: /content/drive/MyDrive/EcoShieldAI/data/raw/train_transaction.csv
Identity yolu: /content/drive/MyDrive/EcoShieldAI/data/raw/train_identity.csv

Dosya kontrolleri:
train_transaction.csv bulundu mu? True
train_identity.csv bulundu mu? True


In [5]:
import time
import pandas as pd

start_time = time.time()

print("Transaction verisi okunuyor...")
transaction = pd.read_csv(
    TRANSACTION_PATH,
    low_memory=True
)

print("Identity verisi okunuyor...")
identity = pd.read_csv(
    IDENTITY_PATH,
    low_memory=True
)

elapsed_time = time.time() - start_time

print("\nOkuma tamamlandı.")
print("Transaction shape:", transaction.shape)
print("Identity shape:", identity.shape)
print(f"Geçen süre: {elapsed_time / 60:.2f} dakika")

Transaction verisi okunuyor...
Identity verisi okunuyor...

Okuma tamamlandı.
Transaction shape: (590540, 394)
Identity shape: (144233, 41)
Geçen süre: 0.66 dakika


In [6]:
print("Transaction ilk 5 satır:")
display(transaction.head())

print("\nTarget dağılımı:")
print(transaction["isFraud"].value_counts())

print("\nTarget oranı:")
print(transaction["isFraud"].value_counts(normalize=True))

print("\nTransactionID duplicate kontrolü:")
print("Transaction duplicate ID:", transaction["TransactionID"].duplicated().sum())
print("Identity duplicate ID:", identity["TransactionID"].duplicated().sum())

Transaction ilk 5 satır:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Target dağılımı:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Target oranı:
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

TransactionID duplicate kontrolü:
Transaction duplicate ID: 0
Identity duplicate ID: 0


In [7]:
import gc
import time

start_time = time.time()

print("Transaction ve identity tabloları birleştiriliyor...")

df = transaction.merge(
    identity,
    on="TransactionID",
    how="left",
    validate="one_to_one"
)

elapsed_time = time.time() - start_time

print("\nBirleştirme tamamlandı.")
print("Transaction satır sayısı:", len(transaction))
print("Birleşik veri shape:", df.shape)
print(f"Geçen süre: {elapsed_time / 60:.2f} dakika")

# Artık ayrı tablolara ihtiyacımız yok
del transaction, identity
gc.collect()

Transaction ve identity tabloları birleştiriliyor...

Birleştirme tamamlandı.
Transaction satır sayısı: 590540
Birleşik veri shape: (590540, 434)
Geçen süre: 0.02 dakika


150

## 2. Veri Temizleme ve Preprocessing

In [ ]:
# Eksik değer analizi - preprocessing kararlarını buna göre vereceğiz
missing = df.isnull().mean().sort_values(ascending=False)
print("En çok eksik değere sahip 20 kolon:")
print((missing.head(20) * 100).round(2).astype(str) + " %")

print("\nHiç eksik değeri olmayan kolon sayısı:", (missing == 0).sum())
print("Eksik değeri %90'dan fazla olan kolon sayısı:", (missing > 0.90).sum())

In [ ]:
def reduce_mem_usage(df, verbose=True):
    """
    Sayısal kolonların dtype'larını küçülterek RAM kullanımını azaltır.
    434 kolonluk bu veri setinde bu adım kritik önem taşıyor.
    """
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    int_columns = df.select_dtypes(include=["int64", "int32"]).columns
    float_columns = df.select_dtypes(include=["float64"]).columns

    for col in int_columns:
        col_min, col_max = df[col].min(), df[col].max()
        if col_min >= 0:
            if col_max < 255:
                df[col] = df[col].astype(np.uint8)
            elif col_max < 65535:
                df[col] = df[col].astype(np.uint16)
            else:
                df[col] = df[col].astype(np.uint32)
        else:
            if col_min > -128 and col_max < 127:
                df[col] = df[col].astype(np.int8)
            elif col_min > -32768 and col_max < 32767:
                df[col] = df[col].astype(np.int16)
            else:
                df[col] = df[col].astype(np.int32)

    for col in float_columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    if verbose:
        print(f"Bellek kullanımı: {start_mem:.1f} MB -> {end_mem:.1f} MB "
              f"(%{100 * (start_mem - end_mem) / start_mem:.1f} azalma)")
    return df


df = reduce_mem_usage(df)
gc.collect()

In [ ]:
# TransactionDT, referans bir zamandan (ilk işlem anı) itibaren geçen saniye sayısı.
# Buradan gün içi saat ve haftanın günü gibi döngüsel zaman özellikleri türetebiliriz.
SECONDS_IN_DAY = 24 * 60 * 60

df["Transaction_hour"] = (df["TransactionDT"] // 3600) % 24
df["Transaction_day"] = (df["TransactionDT"] // SECONDS_IN_DAY) % 7

# TransactionAmt'ın küsürat kısmı - dolandırıcılık işlemlerinde farklı bir dağılım gösterebiliyor
df["TransactionAmt_decimal"] = ((df["TransactionAmt"] - df["TransactionAmt"].astype(int)) * 1000).astype(int)

# Email domain'lerini sadeleştirme (örn. gmail.com, hotmail.co.uk -> gmail, hotmail)
df["P_emaildomain_bin"] = df["P_emaildomain"].str.split(".").str[0]
df["R_emaildomain_bin"] = df["R_emaildomain"].str.split(".").str[0]

print("Yeni özellikler eklendi:")
print(["Transaction_hour", "Transaction_day", "TransactionAmt_decimal",
       "P_emaildomain_bin", "R_emaildomain_bin"])

In [ ]:
# LightGBM kategorik değişkenleri native olarak işleyebiliyor (one-hot'a gerek yok).
# Bu hem RAM'den tasarruf sağlar hem de card1 gibi yüksek kardinaliteli kolonlarda
# one-hot encoding'den çok daha performanslıdır.

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
print(f"Kategorik kolon sayısı: {len(categorical_cols)}")
print(categorical_cols)

for col in categorical_cols:
    df[col] = df[col].astype("category")

## 3. Train / Validation / Test Ayrımı

In [ ]:
# ÖNEMLİ: Fraud detection bir zaman serisi problemi gibi davranır.
# Rastgele train_test_split kullanırsak gelecekteki işlemlerin bilgisi
# eğitim setine sızabilir (temporal data leakage) ve gerçek dünyada
# göremeyeceğimiz kadar iyimser bir skor elde ederiz. Bu yüzden
# TransactionDT'ye göre kronolojik bir ayrım yapıyoruz: model geçmiş
# işlemlerle eğitilip gelecek işlemler üzerinde test edilecek -
# production senaryosunun birebir aynısı.

df = df.sort_values("TransactionDT").reset_index(drop=True)

train_end = int(len(df) * 0.70)
valid_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end]
valid_df = df.iloc[train_end:valid_end]
test_df = df.iloc[valid_end:]

print("Train:", train_df.shape, "| Fraud oranı:", round(train_df["isFraud"].mean(), 4))
print("Valid:", valid_df.shape, "| Fraud oranı:", round(valid_df["isFraud"].mean(), 4))
print("Test :", test_df.shape, "| Fraud oranı:", round(test_df["isFraud"].mean(), 4))

drop_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X_train, y_train = train_df[feature_cols], train_df["isFraud"]
X_valid, y_valid = valid_df[feature_cols], valid_df["isFraud"]
X_test, y_test = test_df[feature_cols], test_df["isFraud"]

del df, train_df, valid_df, test_df
gc.collect()

## 4. Class Imbalance Analizi

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Negatif (fraud olmayan) örnek : {neg}")
print(f"Pozitif (fraud) örnek         : {pos}")
print(f"Dengesizlik oranı             : 1 fraud için {neg / pos:.1f} normal işlem var")
print(f"Önerilen scale_pos_weight     : {scale_pos_weight:.2f}")

## 5. Heavy LightGBM Modelinin Eğitilmesi

"Heavy" burada şu anlama geliyor: düşük learning_rate + yüksek n_estimators
(early stopping ile durduracağız) + geniş num_leaves. Bu, hızlı ama daha az
doğru bir modelden ziyade, biraz daha uzun sürede daha güçlü bir model
elde etmeyi hedefler.

In [ ]:
categorical_features = [c for c in feature_cols if str(X_train[c].dtype) == "category"]

lgb_params = dict(
    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    n_estimators=5000,          # early stopping ile durduracağız, üst limit yüksek
    learning_rate=0.01,         # düşük LR + çok sayıda ağaç = "heavy" model
    num_leaves=256,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

model = LGBMClassifier(**lgb_params)

start_time = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_names=["train", "valid"],
    eval_metric="auc",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)

elapsed_time = time.time() - start_time
print(f"\nEğitim süresi: {elapsed_time / 60:.2f} dakika")
print(f"En iyi iterasyon: {model.best_iteration_}")
print(f"En iyi valid AUC: {model.best_score_['valid']['auc']:.5f}")

## 6. Threshold Tuning

`predict_proba >= 0.5` kuralı, dengesiz veri setlerinde neredeyse hiçbir
zaman optimal değildir. Bunun yerine validation seti üzerinde
precision-recall eğrisini kullanarak F1'i maksimize eden threshold'u
seçiyoruz.

In [ ]:
from sklearn.metrics import precision_recall_curve

valid_proba = model.predict_proba(X_valid)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_valid, valid_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

print(f"F1'i maksimize eden threshold : {best_threshold:.4f}")
print(f"Bu threshold'daki Precision   : {precisions[best_idx]:.4f}")
print(f"Bu threshold'daki Recall      : {recalls[best_idx]:.4f}")
print(f"Bu threshold'daki F1          : {f1_scores[best_idx]:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.plot(thresholds, f1_scores[:-1], label="F1")
plt.axvline(best_threshold, color="gray", linestyle="--", label=f"Seçilen threshold={best_threshold:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Skor")
plt.title("Threshold'a Göre Precision / Recall / F1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Model Metriklerinin Hesaplanması (Test Seti)

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

test_auc = roc_auc_score(y_test, test_proba)
test_ap = average_precision_score(y_test, test_proba)  # PR-AUC: dengesiz veride ROC-AUC'dan daha bilgilendirici
test_precision = precision_score(y_test, test_pred)
test_recall = recall_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)

print("=== Test Seti Sonuçları ===")
print(f"ROC-AUC     : {test_auc:.5f}")
print(f"PR-AUC (AP) : {test_ap:.5f}")
print(f"Precision   : {test_precision:.5f}")
print(f"Recall      : {test_recall:.5f}")
print(f"F1          : {test_f1:.5f}")

cm = confusion_matrix(y_test, test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Fraud"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Test Seti - Confusion Matrix")
plt.show()

## 8. Feature Importance Analizi

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

top_n = 30
plt.figure(figsize=(10, 10))
plt.barh(importance_df["feature"].head(top_n)[::-1], importance_df["importance"].head(top_n)[::-1])
plt.title(f"En Önemli {top_n} Özellik")
plt.xlabel("Importance (split sayısına dayalı)")
plt.tight_layout()
plt.show()

print(importance_df.head(20).to_string(index=False))

## 9. Model ve Sonuçların Kaydedilmesi

In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/EcoShieldAI/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "lightgbm_heavy_v1.joblib"
joblib.dump(model, model_path)

results = {
    "model_params": lgb_params,
    "best_iteration": int(model.best_iteration_),
    "best_threshold": float(best_threshold),
    "metrics": {
        "roc_auc": float(test_auc),
        "pr_auc": float(test_ap),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
    },
    "feature_count": len(feature_cols),
    "train_shape": list(X_train.shape),
    "valid_shape": list(X_valid.shape),
    "test_shape": list(X_test.shape),
}

results_path = OUTPUT_DIR / "results_v1.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Model kaydedildi   :", model_path)
print("Sonuçlar kaydedildi:", results_path)